In [1]:
import pandas as pd
import os

In [2]:
#load datasets
DATA_PATH = 'data/'
train_data = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'))
test_data = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))

print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)


In [3]:
#Check missing values
missing = train_data.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Columns with missing values:\n", missing)

Columns with missing values:
 PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64


In [6]:
# Step 2: Handling missing values safely

# Numerical columns with a clear fill value
train_data['LotFrontage'] = train_data['LotFrontage'].fillna(train_data['LotFrontage'].median())
train_data['MasVnrArea'] = train_data['MasVnrArea'].fillna(0)

# Categorical columns that should be 'None' when missing
none_fill_cols = ['Alley', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature',
                  'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                  'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                  'MasVnrType']

for col in none_fill_cols:
    train_data[col] = train_data[col].fillna('None')

# Basement numeric features (fill 0)
bsmt_numeric_cols = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
                     'BsmtFullBath', 'BsmtHalfBath']
for col in bsmt_numeric_cols:
    train_data[col] = train_data[col].fillna(0)

# Garage numeric column: GarageYrBlt, fill with 0 (or median year if you prefer)
train_data['GarageYrBlt'] = train_data['GarageYrBlt'].fillna(0)

# Electrical has only 1 missing value, fill with mode
train_data['Electrical'] = train_data['Electrical'].fillna(train_data['Electrical'].mode()[0])

In [8]:
#Converting data types if necessary
train_data['MSSubClass'] = train_data['MSSubClass'].astype(str)
train_data['YrSold'] = train_data['YrSold'].astype(str)
train_data['MoSold'] = train_data['MoSold'].astype(str)

In [9]:
#Verifying
print(train_data.isnull().sum().sum()) 

0


In [10]:
# One-hot encode all categorical columns
train_encoded = pd.get_dummies(train_data, drop_first=True)

# Check shape to see how many new columns were created
print(train_encoded.shape)

(1460, 277)


In [11]:
# Target variable
y = train_encoded['SalePrice']

# Features (drop SalePrice)
X = train_encoded.drop('SalePrice', axis=1)

# Optional: save cleaned datasets
train_encoded.to_csv('data/train_encoded.csv', index=False)
X.to_csv('data/X.csv', index=False)
y.to_csv('data/y.csv', index=False)